# LightGBM Ranker - Alpha 5D

Notebook này train mô hình **cross-sectional ranking** để chọn cổ phiếu mạnh hơn thị trường trong 5 phiên tới.

Target chính:
- `alpha_5d = return_5d_stock - return_5d_vnindex`

Model:
- `LightGBM Ranker (lambdarank)`
- group theo `trading_date`
- output là score để chọn `top-k` mỗi ngày


In [ ]:
%pip install -q lightgbm

from google.colab import drive
drive.mount('/content/drive')

import json
import os
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
# === BƯỚC 1: LOAD DATA ===
DATA_PATH = '/content/drive/MyDrive/all_stocks_train.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
df['trading_date'] = pd.to_datetime(df['trading_date'])
df = df.drop_duplicates(subset=['symbol', 'trading_date'], keep='last')
df = df.sort_values(['symbol', 'trading_date']).reset_index(drop=True)

print(f'Raw data: {len(df):,} dòng | {df["symbol"].nunique()} mã')
print(df[['symbol', 'trading_date']].head())


In [ ]:
# === BƯỚC 2: FILTER + MERGE VNINDEX ===
START_DATE = '2021-01-01'
MIN_VOLUME_SMA20 = 100000

vnindex = (
    df[df['symbol'] == 'VNINDEX'][['trading_date', 'close', 'return_5d']]
    .drop_duplicates(subset=['trading_date'], keep='last')
    .sort_values('trading_date')
    .reset_index(drop=True)
)

vnindex['vnindex_momentum_5'] = vnindex['close'].pct_change(5).clip(-0.3, 0.3)
vnindex['vnindex_momentum_20'] = vnindex['close'].pct_change(20).clip(-0.3, 0.3)
vnindex_diff = vnindex['close'].diff()
gain = vnindex_diff.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
loss = (-vnindex_diff.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
rs = gain / loss.replace(0, np.nan)
vnindex['vnindex_rsi'] = 100 - (100 / (1 + rs))
vnindex = vnindex.rename(columns={'return_5d': 'vnindex_return_5d'})

stocks = df[df['symbol'] != 'VNINDEX'].copy()
stocks = stocks[stocks['exchange'].isin(['HOSE', 'HNX'])]
stocks = stocks[stocks['volume_sma_20'] > MIN_VOLUME_SMA20]
stocks = stocks[stocks['trading_date'] >= START_DATE]

stocks = stocks.merge(vnindex, on='trading_date', how='left', validate='many_to_one')
stocks['alpha_5d'] = stocks['return_5d'] - stocks['vnindex_return_5d']

print(f'After filter: {len(stocks):,} dòng | {stocks["symbol"].nunique()} mã')
print(f'VNINDEX rows : {len(vnindex):,}')
print(vnindex[['trading_date', 'vnindex_return_5d', 'vnindex_momentum_5', 'vnindex_momentum_20', 'vnindex_rsi']].head(3))
print(f'Alpha rows    : {stocks["alpha_5d"].notna().sum():,}')


In [ ]:
# === BƯỚC 3: FEATURE ENGINEERING ===
BASE_FEATURES = [
    'price_vs_sma20', 'price_vs_sma50', 'sma20_vs_sma50',
    'price_momentum_5', 'price_momentum_10', 'price_momentum_20',
    'macd_norm', 'macd_hist_norm',
    'adx_14', 'di_plus', 'di_minus',
    'rsi_14', 'stoch_k', 'stoch_d', 'williams_r',
    'bb_pct_b', 'bb_width_norm', 'atr_pct',
    'cmf_20', 'volume_ratio', 'volume_momentum_5',
    'candle_body_pct', 'candle_upper_pct', 'candle_lower_pct',
    'is_doji', 'is_hammer', 'is_bull_engulfing', 'is_bear_engulfing',
    'is_morning_star', 'is_evening_star',
    'vnindex_momentum_5', 'vnindex_momentum_20', 'vnindex_rsi',
]

stocks['rel_momentum_5'] = (stocks['price_momentum_5'] - stocks['vnindex_momentum_5']).clip(-0.5, 0.5)
stocks['rel_momentum_20'] = (stocks['price_momentum_20'] - stocks['vnindex_momentum_20']).clip(-0.5, 0.5)
stocks['rsi_vs_vnindex'] = (stocks['rsi_14'] - stocks['vnindex_rsi']).clip(-100, 100)

RANK_SOURCE_COLS = [
    'price_momentum_5', 'price_momentum_10', 'price_momentum_20',
    'rsi_14', 'stoch_k', 'stoch_d', 'williams_r',
    'bb_pct_b', 'volume_ratio', 'cmf_20',
    'rel_momentum_5', 'rel_momentum_20', 'rsi_vs_vnindex',
]

for col in RANK_SOURCE_COLS:
    stocks[f'{col}_rank_pct'] = stocks.groupby('trading_date')[col].rank(pct=True)

FEATURES = BASE_FEATURES + [
    'rel_momentum_5', 'rel_momentum_20', 'rsi_vs_vnindex'
] + [f'{col}_rank_pct' for col in RANK_SOURCE_COLS]

stocks = stocks.dropna(subset=['alpha_5d'])
stocks[FEATURES] = stocks.groupby('symbol')[FEATURES].ffill()
stocks = stocks.dropna(subset=FEATURES)
stocks['alpha_5d'] = stocks['alpha_5d'].astype(float).clip(-0.30, 0.30)

print(f'Tổng features: {len(FEATURES)}')
print(f'Tỷ lệ alpha_5d > 0: {(stocks["alpha_5d"] > 0).mean():.2%}')
print(f'Mean alpha_5d    : {stocks["alpha_5d"].mean():.4%}')
print(f'Median alpha_5d  : {stocks["alpha_5d"].median():.4%}')
print(f'Usable rows      : {len(stocks):,}')


In [ ]:
# === BƯỚC 4: BUILD RANK LABEL + SPLIT ===
TEST_CUTOFF = pd.Timestamp('2025-10-01')
VALID_DAYS = 90
VALID_CUTOFF = TEST_CUTOFF - pd.Timedelta(days=VALID_DAYS)
MIN_GROUP_SIZE = 150
N_LABEL_BINS = 5

group_sizes = stocks.groupby('trading_date')['symbol'].transform('size')
stocks = stocks[group_sizes >= MIN_GROUP_SIZE].copy()

alpha_rank_pct = stocks.groupby('trading_date')['alpha_5d'].rank(pct=True, method='first')
stocks['rank_label'] = np.floor(alpha_rank_pct * N_LABEL_BINS).clip(0, N_LABEL_BINS - 1).astype(int)

train = stocks[stocks['trading_date'] < VALID_CUTOFF].copy()
valid = stocks[(stocks['trading_date'] >= VALID_CUTOFF) & (stocks['trading_date'] < TEST_CUTOFF)].copy()
test  = stocks[stocks['trading_date'] >= TEST_CUTOFF].copy()

def make_group(frame):
    return frame.groupby('trading_date').size().tolist()

X_train = train[FEATURES]
y_train = train['rank_label']
group_train = make_group(train)

X_valid = valid[FEATURES]
y_valid = valid['rank_label']
group_valid = make_group(valid)

X_test = test[FEATURES]
y_test = test['rank_label']
group_test = make_group(test)

for name, frame in [('Train', train), ('Valid', valid), ('Test', test)]:
    daily_size = frame.groupby('trading_date').size()
    print(f'\n{name}: {len(frame):,} rows | {frame["trading_date"].nunique()} days')
    print(f'  Mean alpha    : {frame["alpha_5d"].mean():.4%}')
    print(f'  Alpha > 0     : {(frame["alpha_5d"] > 0).mean():.2%}')
    print(f'  Group min/med : {daily_size.min()} / {int(daily_size.median())}')


In [ ]:
# === BƯỚC 5: TRAIN LIGHTGBM RANKER ===
model = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    ndcg_eval_at=[10, 20, 50],
    learning_rate=0.03,
    n_estimators=1500,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.2,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    group=group_train,
    eval_set=[(X_valid, y_valid)],
    eval_group=[group_valid],
    eval_at=[10, 20, 50],
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100),
    ],
)

print('Best iteration:', model.best_iteration_)


In [ ]:
# === BƯỚC 6: ĐÁNH GIÁ ===
valid_score = model.predict(X_valid)
test_score = model.predict(X_test)

valid_eval = valid[['symbol', 'trading_date', 'alpha_5d', 'return_5d']].copy()
valid_eval['score'] = valid_score
test_eval = test[['symbol', 'trading_date', 'alpha_5d', 'return_5d', 'vnindex_return_5d']].copy()
test_eval['score'] = test_score

def evaluate_topk(frame, ks=(10, 20, 50), ascending=False):
    rows = []
    for k in ks:
        picks = (
            frame.sort_values(['trading_date', 'score'], ascending=[True, ascending])
                 .groupby('trading_date', group_keys=False)
                 .head(k)
        )
        rows.append({
            'top_k': k,
            'samples': len(picks),
            'days': picks['trading_date'].nunique(),
            'avg_alpha': picks['alpha_5d'].mean(),
            'alpha_win_rate': (picks['alpha_5d'] > 0).mean(),
            'avg_stock_return': picks['return_5d'].mean(),
            'stock_win_rate': (picks['return_5d'] > 0).mean(),
        })
    return pd.DataFrame(rows)

valid_topk = evaluate_topk(valid_eval, ascending=False)
test_topk = evaluate_topk(test_eval, ascending=False)
test_bottomk = evaluate_topk(test_eval, ascending=True)

universe_test = test.groupby('trading_date')[['alpha_5d', 'return_5d']].mean().mean()

print('\n' + '=' * 72)
print('LIGHTGBM RANKER REPORT')
print('=' * 72)
print(f'Valid avg daily alpha universe : {valid.groupby("trading_date")["alpha_5d"].mean().mean():.4%}')
print(f'Test avg daily alpha universe  : {universe_test["alpha_5d"]:.4%}')
print(f'Test avg daily return universe : {universe_test["return_5d"]:.4%}')

print('\nTop-k long trên validation:')
print(valid_topk.to_string(index=False))

print('\nTop-k long trên test:')
print(test_topk.to_string(index=False))

print('\nTop-k yếu nhất trên test:')
print(test_bottomk.to_string(index=False))

fi = pd.DataFrame({
    'feature': FEATURES,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

print('\nTop 20 features:')
print(fi.head(20).to_string(index=False))

plt.figure(figsize=(10, 8))
plt.barh(fi.head(20)['feature'][::-1], fi.head(20)['importance'][::-1])
plt.title('Top 20 Feature Importance - LightGBM Ranker')
plt.tight_layout()
plt.show()


In [ ]:
# === BƯỚC 7: LƯU MODEL ===
OUTPUT_DIR = '/content/drive/MyDrive/StockData/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = f'{OUTPUT_DIR}/lgbm_alpha_5d_ranker.txt'
META_PATH = f'{OUTPUT_DIR}/lgbm_alpha_5d_features.json'
FI_PATH = f'{OUTPUT_DIR}/lgbm_alpha_5d_feature_importance.csv'

model.booster_.save_model(MODEL_PATH)
with open(META_PATH, 'w', encoding='utf-8') as f:
    json.dump({
        'model_type': 'lightgbm_ranker',
        'target': 'alpha_5d',
        'features': FEATURES,
        'min_group_size': MIN_GROUP_SIZE,
        'n_label_bins': N_LABEL_BINS,
    }, f, indent=2, ensure_ascii=False)
fi.to_csv(FI_PATH, index=False)

print('Saved:')
print(MODEL_PATH)
print(META_PATH)
print(FI_PATH)
